In [ ]:
# ============================================================
# Imports
# ============================================================
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ============================================================
# Font settings
# ============================================================
TICK_FONTSIZE = 15
LABEL_FONTSIZE = 16
TITLE_FONTSIZE = 16
ANNOT_FONTSIZE = 11
CBAR_TICK_FONTSIZE = 14
CBAR_LABEL_FONTSIZE = 15

plt.rcParams.update({
    "font.size": 14,
    "axes.labelsize": LABEL_FONTSIZE,
    "axes.titlesize": TITLE_FONTSIZE,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# ============================================================
# Parameters
# ============================================================
a, b, e, alpha, epsion, mu, c = [
    5.24073718,
    6.10303089,
    0.85355013,
    0.32682177,
    -0.00968044,
    0.84661793,
    -0.27288932,
]

beta = alpha

A = np.exp(a)
B = np.exp(b)
E = np.exp(e)

# ============================================================
# Optimal allocation
# ============================================================
def optimal_N(C, s):
    a_ = beta / (alpha + beta)
    A_t = (s**epsion + c * (1 - s)**mu) * A
    G = ((alpha * A_t) / (beta * B)) ** (1 / (alpha + beta))
    return G * (C / 6) ** a_

def optimal_D(C, s):
    b_ = alpha / (alpha + beta)
    A_t = (s**epsion + c * (1 - s)**mu) * A
    G = ((alpha * A_t) / (beta * B)) ** (1 / (alpha + beta))
    return (1 / G) * (C / 6) ** b_

def D_to_C(D, s):
    b_ = alpha / (alpha + beta)
    A_t = (s**epsion + c * (1 - s)**mu) * A
    G = ((alpha * A_t) / (beta * B)) ** (1 / (alpha + beta))
    return ((G * D) ** (1 / b_)) * 6

# ============================================================
# Scaling law
# ============================================================
def scaling_law(N, U, D, S):
#     a2, b2, e2, alpha2, epsion2, mu2, c2 = stage2_params
#     beta2 = alpha2

#     gamma1_2, gamma2_2 = stage3_params
#     c11_2, c12_2, c21_2, c22_2 = stage4_params

    a2, b2, e2, alpha2, epsion2, mu2, c2 = [
    5.24073718,
    6.10303089,
    0.85355013,
    0.32682177,
    -0.00968044,
    0.84661793,
    -0.27288932,
]

    beta2 = alpha2

    gamma1_2, gamma2_2 = [
        11.08763712,
        4.40474882,
    ]

    c11_2, c12_2, c21_2, c22_2 = [
        1.90420893,
        1.82159486,
        2.79936732,
        1.36557887,
    ]

    A2, B2, E2 = np.exp(a2), np.exp(b2), np.exp(e2)

    UN = np.minimum(N, optimal_N(D_to_C(U, S), S))

    rn = np.maximum((N / UN) - 1, 0)
    rd = np.maximum((D / U) - 1, 0)

    n_star = gamma1_2 * (1 + (1 - S) * c11_2 - c21_2 * (1 - S) ** 2)
    d_star = gamma2_2 * (1 + (1 - S) * c12_2 - c22_2 * (1 - S) ** 2)

    N_eff = (UN + UN * n_star * (1 - np.exp(-rn / n_star))) ** alpha2
    D_eff = (U + U * d_star * (1 - np.exp(-rd / d_star))) ** beta2

    return (
        E2
        + A2 * (S**epsion2 + c2 * (1 - S) ** mu2) / N_eff
        + B2 / D_eff
    )

# ============================================================
# Best loss at fixed S & FLOPs
# ============================================================
def best_loss_per_F_at_S(
    S,
    toks,
    sizes,
    F_list,
    UT,
    scale_min=1.0001,
    scale_max=10.0,
    num_scale=400,
):
    out = {}
    scales = np.linspace(scale_min, scale_max, num_scale)

    for t, s, F in zip(toks, sizes, F_list):
        best = scaling_law(s, min(UT, t), t, S)

        for r in scales:
            best = min(
                best,
                scaling_law(s / r, min(UT, t * r), t * r, S),
                scaling_law(s * r, min(UT, t / r), t / r, S),
            )

        out[float(F)] = best

    return out

# ============================================================
# Smart annotation helper
# ============================================================
def smart_annotate(
    ax,
    xy,
    text,
    placed_bboxes,
    candidates,
    bbox_style,
    arrow_style,
    fontsize=ANNOT_FONTSIZE,
):
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    ann = None
    last_bbox = None

    for dx, dy, ha, va in candidates:
        if ann is not None:
            ann.remove()

        ann = ax.annotate(
            text,
            xy=xy,
            xytext=(dx, dy),
            textcoords="offset points",
            ha=ha,
            va=va,
            fontsize=fontsize,
            bbox=bbox_style,
            arrowprops=arrow_style,
            annotation_clip=False,
            zorder=20,
        )

        fig.canvas.draw()
        bbox = ann.get_window_extent(renderer=renderer).expanded(1.08, 1.15)
        last_bbox = bbox

        overlap = any(bbox.overlaps(prev_bbox) for prev_bbox in placed_bboxes)

        if not overlap:
            placed_bboxes.append(bbox)
            return ann

    placed_bboxes.append(last_bbox)
    return ann

# ============================================================
# Globals
# ============================================================
random.seed(42)
np.random.seed(42)

S_vals = np.linspace(0.05, 1.0, 120)
UT_list = [1.3e9, 1.3e10, 1.3e11]

NUM_FLOP_LINES = 18

def flops_grid_from_UT(UT, n=NUM_FLOP_LINES):
    base = UT * np.log10(UT / 1e7)

    fmin_scale = {
        1.3e9: 1.0,
        1.3e10: 0.5,
        1.3e11: 0.4,
    }[UT]

    F_min = 6 * base**2 * fmin_scale
    F_max = 75 * base**2 * fmin_scale

    return np.logspace(
        np.log10(F_min),
        np.log10(F_max),
        n,
    )

# ============================================================
# Plot per UT
# ============================================================
def plot_for_UT(ax, UT):
    F_list = flops_grid_from_UT(UT, n=NUM_FLOP_LINES)
    curves = {float(F): [] for F in F_list}

    # ----- compute curves -----
    for S in S_vals:
        toks = np.array([optimal_D(F, S) for F in F_list])
        sizes = np.array([optimal_N(F, S) for F in F_list])

        res = best_loss_per_F_at_S(
            S,
            toks,
            sizes,
            F_list,
            UT,
        )

        for F in F_list:
            curves[float(F)].append(res[float(F)])

    # ----- dense global optimum -----
    S_dense = 1.0
    target_loss = np.inf
    F_dense_opt = None

    for F in F_list:
        toks = np.array([optimal_D(F, S_dense)])
        sizes = np.array([optimal_N(F, S_dense)])

        loss_F = best_loss_per_F_at_S(
            S_dense,
            toks,
            sizes,
            [float(F)],
            UT,
        )[float(F)]

        if loss_F < target_loss:
            target_loss = loss_F
            F_dense_opt = float(F)

    # ----- operating point -----
    F_operating = None
    best_S = None

    for F in sorted(F_list):
        y = np.array(curves[float(F)])
        ok = y <= target_loss

        if np.any(ok):
            F_operating = float(F)
            best_S = S_vals[ok].min()
            break

    compute_gain = F_dense_opt / F_operating

    # ----- global minimum loss -----
    global_min_loss = np.inf
    global_min_S = None
    global_f = None

    for F in F_list:
        y = np.array(curves[float(F)])
        i = np.argmin(y)

        if y[i] < global_min_loss:
            global_min_loss = y[i]
            global_min_S = S_vals[i]
            global_f = float(F)

    # ----- plotting FLOPs curves -----
    logF = np.log10(F_list)
    norm = Normalize(logF.min(), logF.max())
    cmap = mpl.colormaps["gist_heat"]

    for F in F_list:
        ax.plot(
            1 - S_vals,
            curves[float(F)],
            "--",
            lw=2,
            color=cmap(norm(np.log10(F))),
        )

    # ----- reference lines -----
    ax.axhline(target_loss, color="black", lw=1.2)
    ax.axvline(1 - best_S, color="black", lw=1.2)

    # ----- key points -----
    x_dense, y_dense = 0.0, target_loss
    x_oper, y_oper = 1 - best_S, target_loss
    x_global, y_global = 1 - global_min_S, global_min_loss

    N_sparse = optimal_N(F_operating, best_S)
    N_dense = optimal_N(F_dense_opt, 1.0)
    N_sparse_global = optimal_N(global_f, global_min_S)

    # ----- markers -----
    ax.scatter(
        x_dense,
        y_dense,
        s=90,
        marker="o",
        color="dodgerblue",
        edgecolors="black",
        linewidths=1.0,
        zorder=8,
    )

    ax.scatter(
        x_oper,
        y_oper,
        s=150,
        marker="*",
        color="red",
        edgecolors="black",
        linewidths=1.0,
        zorder=9,
    )

    ax.scatter(
        x_global,
        y_global,
        s=170,
        marker="*",
        color="gold",
        edgecolors="black",
        linewidths=1.2,
        zorder=9,
    )

    # ----- annotation style -----
    bbox_style = dict(
        boxstyle="round,pad=0.25",
        fc="white",
        ec="black",
        alpha=0.95,
    )

    arrow_style = dict(
        arrowstyle="->",
        lw=1.0,
        color="black",
        shrinkA=0,
        shrinkB=4,
    )

    placed_bboxes = []

    dense_candidates = [
        (12, 14, "left", "bottom"),
        (12, 30, "left", "bottom"),
        (45, 14, "left", "bottom"),
        (45, 30, "left", "bottom"),
        (12, -10, "left", "top"),
        (30, -15, "left", "top"),
    ]

    oper_candidates = [
        (0, -14, "center", "top"),
        (35, -10, "left", "top"),
        (-35, -10, "right", "top"),
        (0, 14, "center", "bottom"),
        (35, 14, "left", "bottom"),
        (-35, 14, "right", "bottom"),
        (55, 0, "left", "center"),
        (-55, 0, "right", "center"),
    ]

    global_candidates = [
        (0, 18, "center", "bottom"),
        (-20, 18, "right", "bottom"),
        (20, 18, "left", "bottom"),
        (0, -16, "center", "top"),
        (-45, 12, "right", "bottom"),
        (45, 12, "left", "bottom"),
    ]

    # ----- annotations -----
    smart_annotate(
        ax,
        (x_dense, y_dense),
        f"{compute_gain:.1f}× compute\nN = {N_dense / 1e9:.2f}B",
        placed_bboxes,
        dense_candidates,
        bbox_style,
        arrow_style,
        fontsize=ANNOT_FONTSIZE,
    )

    smart_annotate(
        ax,
        (x_oper, y_oper),
        f"N = {N_sparse / 1e9:.2f}B",
        placed_bboxes,
        oper_candidates,
        bbox_style,
        arrow_style,
        fontsize=ANNOT_FONTSIZE,
    )

    smart_annotate(
        ax,
        (x_global, y_global),
        f"N = {N_sparse_global / 1e9:.2f}B",
        placed_bboxes,
        global_candidates,
        bbox_style,
        arrow_style,
        fontsize=ANNOT_FONTSIZE,
    )

    # ----- cosmetics -----
    ax.set_title(
        f"Ud = {UT / 1e9:.1f}B",
        fontsize=TITLE_FONTSIZE,
        fontweight="bold",
    )

    ax.set_xlabel(
        "Sparsity",
        fontsize=LABEL_FONTSIZE,
        fontweight="bold",
    )

    ax.grid(alpha=0.25, linestyle=":")

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_FONTSIZE,
        width=1.8,
        length=5,
    )

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    cbar = plt.colorbar(
        sm,
        ax=ax,
        pad=0.01,
        fraction=0.045,
        label=r"$\log_{10}(\mathrm{Train\ FLOPs})$",
    )

    cbar.ax.tick_params(
        labelsize=CBAR_TICK_FONTSIZE,
        width=1.5,
        length=4,
    )

    cbar.ax.yaxis.label.set_size(CBAR_LABEL_FONTSIZE)

# ============================================================
# Main
# ============================================================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4),
    constrained_layout=True,
)

for ax, UT in zip(axes, UT_list):
    plot_for_UT(ax, UT)

axes[0].set_ylabel(
    "Validation loss",
    fontsize=LABEL_FONTSIZE,
    fontweight="bold",
)

for ax in axes.flat:
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=TICK_FONTSIZE,
        width=2,
        length=5,
    )

plt.savefig("optimal_sparsity.pdf", bbox_inches="tight")
plt.savefig("optimal_sparsity.png", dpi=300, bbox_inches="tight")
plt.savefig("optimal_sparsity.svg", format="svg", bbox_inches="tight")

plt.show()